# 03 — Golden Evaluation Set

Build a human-verified evaluation set on top of the **frozen** taxonomy
(`configs/intents.yaml`, SHA-256 pinned).

**This notebook does NOT build a classifier, retrieval, or replies.**

## Hard rules

1. **Disjoint data.** No `root_id` from `coverage_labeling_sheet.csv`,
   `intent_examples.csv`, or `excluded_uninformative_messages.csv` may appear.
2. **Every candidate must be labeled.** No skipped rows; the validation
   cell asserts 100% coverage of the annotation pool.
3. **Human verification.** Every kept row has a human-confirmed
   `true_intent`. Suggested labels are annotation assistance only and are
   never treated as ground truth.
4. **Frozen taxonomy.** The taxonomy SHA-256 is recorded in the
   golden-set manifest. Drift invalidates the set.
5. **Evaluation unit:** the first inbound customer message of a GWRHelp
   thread. Not full conversations.
6. **Coverage-oriented, not prevalence-representative.** Targeted
   supplementation ensures adequate representation of low-frequency
   intents. The `_source` column (natural vs targeted) is preserved so
   the classifier report can distinguish the two slices.
7. **Test lock.** The test split is for final reported metrics only. No
   classifier, prompt, retrieval, or escalation-rule tuning against test.

## Pipeline


In [3]:
# ============ CELL 2: Setup + verify frozen taxonomy ============
from pathlib import Path
import os, re, json, hashlib, sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

def find_project_root():
    markers = [".git", "pyproject.toml"]
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if any((parent / m).exists() for m in markers):
            return parent
    raise RuntimeError(f"Could not locate ResolveIQ project root from {p}")

PROJECT_ROOT = find_project_root()
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BRAND        = "GWRHelp"
TAX_DIR      = PROJECT_ROOT / "runs" / "taxonomy"
GS_DIR       = PROJECT_ROOT / "runs" / "golden_set"
CONFIG_DIR   = PROJECT_ROOT / "configs"
EVAL_DIR     = PROJECT_ROOT / "evaluation"

for d in (GS_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

FIRST_INBOUND = PROJECT_ROOT / "runs" / "brand_selection" / "first_inbound_per_thread.pkl"
INTENTS_YAML  = CONFIG_DIR / "intents.yaml"
INTENTS_SHA   = TAX_DIR / "intents.yaml.sha256"
FROZEN_YAML   = TAX_DIR / "frozen" / "intents_v1.yaml"

for p in (FIRST_INBOUND, INTENTS_YAML, INTENTS_SHA, FROZEN_YAML):
    assert p.exists(), f"Missing required input: {p}"

yaml_bytes = INTENTS_YAML.read_bytes()
sha        = hashlib.sha256(yaml_bytes).hexdigest()
recorded   = INTENTS_SHA.read_text().strip()
assert sha == recorded, f"Taxonomy SHA mismatch:\n  computed={sha}\n  recorded={recorded}"
assert FROZEN_YAML.read_bytes() == yaml_bytes, "Frozen copy differs from configs/intents.yaml"

tax = yaml.safe_load(yaml_bytes)
assert tax.get("draft") is False, "Taxonomy is still marked draft"
INTENT_NAMES       = [i["name"] for i in tax["intents"]]
INTENT_DEFS        = {i["name"]: i for i in tax["intents"]}
OPERATIONAL_INTENTS = [n for n in INTENT_NAMES if n not in ("other", "ambiguous")]

print(f"PROJECT_ROOT:    {PROJECT_ROOT}")
print(f"Taxonomy SHA:    {sha[:16]}...")
print(f"Intents ({len(INTENT_NAMES)}): {', '.join(INTENT_NAMES)}")
print(f"Operational:     {len(OPERATIONAL_INTENTS)}")

PROJECT_ROOT:    D:\CODIN PLAYGROUND\ML-AI\ResolveIQ
Taxonomy SHA:    7a05e4af68a50981...
Intents (11): delay_compensation, refund_request, booking_issue, timetable_info, service_disruption, seat_reservation, lost_property, on_board_issue, praise_or_chatter, other, ambiguous
Operational:     9


In [4]:
# ============ CELL 3: Candidate pool with exclusions and text dedup ============
df  = pd.read_pickle(FIRST_INBOUND)
gwr = df[df["brand_author_id"] == BRAND].copy()

excluded_uninf = pd.read_csv(TAX_DIR / "excluded_uninformative_messages.csv")
coverage_sheet = pd.read_csv(TAX_DIR / "coverage_labeling_sheet.csv")
examples_sheet = pd.read_csv(TAX_DIR / "intent_examples.csv")

exclude_ids  = set(excluded_uninf["root_id"].astype(int))
exclude_ids |= set(coverage_sheet["root_id"].astype(int))
exclude_ids |= set(examples_sheet["root_id"].astype(int))

print(f"Corpus:                    {len(gwr):,}")
print(f"Excluded (uninformative):  {len(excluded_uninf):,}")
print(f"Excluded (coverage set):   {coverage_sheet['root_id'].nunique():,}")
print(f"Excluded (examples set):   {examples_sheet['root_id'].nunique():,}")
print(f"Total unique excluded:     {len(exclude_ids):,}")

pool = gwr[~gwr["root_id"].astype(int).isin(exclude_ids)].copy()
pool = pool[pool["customer_text"].str.strip().str.len() >= 5].reset_index(drop=True)

# --- Text-level dedup: Twitter data contains repeated "Thanks!", "Great service", etc. ---
pool["_text_norm"] = (
    pool["customer_text"].astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
)
before = len(pool)
pool = pool.drop_duplicates(subset=["_text_norm"], keep="first").reset_index(drop=True)
after = len(pool)

print(f"\nAfter text dedup:          {after:,}  "
      f"({before - after:,} duplicate messages removed)")

assert len(pool) >= 400, f"Pool too small ({len(pool)}) — check exclusions"

Corpus:                    1,612
Excluded (uninformative):  2
Excluded (coverage set):   300
Excluded (examples set):   95
Total unique excluded:     377

After text dedup:          1,236  (0 duplicate messages removed)


In [5]:
# ============ CELL 4: Natural + rare-intent targeted sampling ============
N_NATURAL            = 250
N_TARGETED_MAX       = 60
MIN_PER_INTENT_EST   = 20   # target estimate per operational intent after supplement

# --- Natural sample ---
natural = pool.sample(n=min(N_NATURAL, len(pool)), random_state=RANDOM_STATE).copy()
natural["_source"] = "natural"
taken_ids = set(natural["root_id"].astype(int))

# --- Probe-based rough estimate of intent counts in the natural sample ---
cand_intents = pd.read_csv(TAX_DIR / "candidate_intents.csv")
probes = {r["intent"]: r["probe_regex"]
          for _, r in cand_intents.iterrows()
          if isinstance(r["probe_regex"], str) and r["probe_regex"]}

probe_est = {}
for intent in OPERATIONAL_INTENTS:
    rx = probes.get(intent, "")
    if not rx:
        probe_est[intent] = 0
        continue
    probe_est[intent] = int(
        natural["customer_text"].str.contains(rx, case=False, na=False, regex=True).sum()
    )

needed = {}
for intent in OPERATIONAL_INTENTS:
    est = probe_est.get(intent, 0)
    if est < MIN_PER_INTENT_EST:
        needed[intent] = MIN_PER_INTENT_EST - est

print("Probe-based natural estimates (rough, not labels):")
for n, c in sorted(probe_est.items(), key=lambda x: -x[1]):
    print(f"  {n:24s} {c:>4d}")
print(f"\nIntents needing supplement: {needed}")
print(f"Total targeted budget:      {sum(needed.values())} (cap {N_TARGETED_MAX})")

# --- Targeted supplement ---
targeted_rows   = []
remaining_budget = N_TARGETED_MAX
for intent, n_want in sorted(needed.items(), key=lambda x: -x[1]):
    if remaining_budget <= 0:
        break
    rx = probes.get(intent, "")
    if not rx:
        continue
    sub = pool[
        (~pool["root_id"].astype(int).isin(taken_ids)) &
        (pool["customer_text"].str.contains(rx, case=False, na=False, regex=True))
    ]
    if len(sub) == 0:
        continue
    n_take = min(n_want, remaining_budget, len(sub))
    s = sub.sample(n=n_take, random_state=RANDOM_STATE).copy()
    s["_source"] = f"targeted:{intent}"
    targeted_rows.append(s)
    taken_ids |= set(s["root_id"].astype(int))
    remaining_budget -= n_take

targeted = (pd.concat(targeted_rows, ignore_index=True)
            if targeted_rows else pd.DataFrame(columns=natural.columns))

candidates = (pd.concat([natural, targeted], ignore_index=True)
                .drop_duplicates(subset=["root_id"])
                .reset_index(drop=True))

print(f"\nNatural candidates:  {len(natural)}")
print(f"Targeted candidates: {len(targeted)}")
print(f"Annotation pool:     {len(candidates)}")
print(f"\nSource distribution:")
print(candidates["_source"].value_counts().to_string())

Probe-based natural estimates (rough, not labels):
  praise_or_chatter          24
  service_disruption         22
  refund_request              4
  timetable_info              4
  booking_issue               3
  delay_compensation          2
  seat_reservation            2
  on_board_issue              1
  lost_property               0

Intents needing supplement: {'delay_compensation': 18, 'refund_request': 16, 'booking_issue': 17, 'timetable_info': 16, 'seat_reservation': 18, 'lost_property': 20, 'on_board_issue': 19}
Total targeted budget:      124 (cap 60)

Natural candidates:  250
Targeted candidates: 60
Annotation pool:     310

Source distribution:
_source
natural                        250
targeted:seat_reservation       17
targeted:refund_request         16
targeted:booking_issue          14
targeted:lost_property           6
targeted:on_board_issue          3
targeted:timetable_info          3
targeted:delay_compensation      1


C:\Users\adity\AppData\Local\Temp\ipykernel_20552\599344113.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  natural["customer_text"].str.contains(rx, case=False, na=False, regex=True).sum()
C:\Users\adity\AppData\Local\Temp\ipykernel_20552\599344113.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (pool["customer_text"].str.contains(rx, case=False, na=False, regex=True))
C:\Users\adity\AppData\Local\Temp\ipykernel_20552\599344113.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (pool["customer_text"].str.contains(rx, case=False, na=False, regex=True))
C:\Users\adity\AppData\Local\Temp\ipykernel_20552\599344113.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actu

In [6]:
# ============ CELL 5: Suggested labels via TF-IDF centroid ============
# SUGGESTIONS ARE SCAFFOLDING ONLY. Never treated as ground truth.
# Never suggests `ambiguous` — that is a true class the human assigns.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

URL_RE  = re.compile(r"https?://\S+|www\.\S+")
MEN_RE  = re.compile(r"(?<!\w)@\w+")
WS_RE   = re.compile(r"\s+")

def _norm(t):
    t = str(t).lower()
    t = URL_RE.sub(" ", t); t = MEN_RE.sub(" ", t); t = WS_RE.sub(" ", t)
    return t.strip()

ex = pd.read_csv(TAX_DIR / "intent_examples.csv")
ex = ex[ex["_keep"].astype(str).str.lower() == "y"]
intent_docs = ex.groupby("intent")["customer_text"].apply(lambda s: " ".join(s.map(_norm)))

vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
ref_matrix = vec.fit_transform(intent_docs.tolist())
ref_names  = intent_docs.index.tolist()

X_cand = vec.transform(candidates["customer_text"].map(_norm).tolist())
sims   = cosine_similarity(X_cand, ref_matrix)

top1_idx  = sims.argmax(axis=1)
top1_sim  = sims[np.arange(len(sims)), top1_idx]
top1_name = [ref_names[i] for i in top1_idx]

# Low confidence → suggest "other" (never "ambiguous")
LOW_SIM_THRESHOLD = 0.10
suggested = [
    ("other" if s < LOW_SIM_THRESHOLD else name)
    for name, s in zip(top1_name, top1_sim)
]

candidates["suggested_intent"]      = suggested
candidates["suggestion_confidence"] = np.round(top1_sim, 3)

print("Suggested label distribution (scaffold — human confirms every row):")
print(candidates["suggested_intent"].value_counts().to_string())

Suggested label distribution (scaffold — human confirms every row):
suggested_intent
other                 94
timetable_info        36
seat_reservation      32
delay_compensation    28
booking_issue         26
service_disruption    24
refund_request        24
on_board_issue        19
lost_property         18
praise_or_chatter      9


In [7]:
# ============ CELL 6: Export labeling sheet ============
LABEL_SHEET = GS_DIR / "labeling_sheet.csv"

if LABEL_SHEET.exists():
    print(f"Keeping existing {LABEL_SHEET}.")
    print("Delete it explicitly to regenerate.")
else:
    out = candidates[[
        "root_id", "customer_text",
        "suggested_intent", "suggestion_confidence", "_source",
    ]].copy()

    out["true_intent"] = ""
    out["confidence"]  = ""     # high | medium | low
    out["rationale"]   = ""     # required for ambiguous + overwrites
    out["_labeler"]    = ""
    out["_labeled_at"] = ""

    out = out.sort_values("root_id").reset_index(drop=True)
    out.to_csv(LABEL_SHEET, index=False, quoting=1)

    print(f"Wrote: {LABEL_SHEET}")
    print(f"Rows:  {len(out)}")
    print("\nNext: open the CSV, review each row, set true_intent.")
    print("EVERY row must be labeled. See Cell 7 for guidelines.")

Keeping existing D:\CODIN PLAYGROUND\ML-AI\ResolveIQ\runs\golden_set\labeling_sheet.csv.
Delete it explicitly to regenerate.


## Annotation guidelines

**Every row must have a `true_intent` from the frozen taxonomy.** No skipping.

### Two orthogonal fields: confidence vs intent

These are **separate concepts**. Do not conflate them.

| Field | Meaning |
|---|---|
| `confidence` | How sure YOU are about the chosen label |
| `true_intent` | The actual class |

**`confidence` values:**
- **high** — clear-cut, no hesitation.
- **medium** — one reasonable alternative is plausible.
- **low** — genuinely uncertain, **but you still believe one intent is more likely than the other**.

**`ambiguous` is a true_intent value, not a confidence level.**

Use `true_intent = "ambiguous"` only when two intents are **genuinely equally plausible**
— you cannot reasonably prefer one over the other.
If you *can* prefer one, use that one with `confidence=low`.

### Tie-break rules

**`delay_compensation` vs `refund_request`** — Specific delayed service
AND asks how to claim / whether they qualify → `delay_compensation`. Asks
for money back on a ticket without mentioning Delay Repay → `refund_request`.
Both present? Use the *dominant ask* (usually the last explicit request).

**`delay_compensation` vs `service_disruption`** — Live disruption affecting
travel *now* or imminently → `service_disruption`. Retrospective delay
already experienced, with compensation ask → `delay_compensation`.

**`on_board_issue` vs `seat_reservation`** — Overcrowding, cleanliness,
temperature, staff conduct → `on_board_issue`. Specific seat/coach
reservation as the *subject* → `seat_reservation`.

**`on_board_issue` vs `service_disruption`** — Cancellation is the topic →
`service_disruption`. Onboard conditions on a train that ran → `on_board_issue`.

**`booking_issue` vs `refund_request`** — Problem during *booking* (website,
payment, confirmation) → `booking_issue`. Post-purchase money-back ask →
`refund_request`.

**`lost_property` vs `on_board_issue`** — Item left on train/station →
`lost_property`. Complaint about conditions, not an item → `on_board_issue`.

### `other` — nothing fits

Station facilities, generic rants, brand commentary, messages with no
customer problem. Assign `other` rather than force a fit.

### `rationale`

**Required** for:
- every `ambiguous` row
- every row where you *changed* `suggested_intent` to a different operational intent

One sentence. Examples:
- "Cancellation dominated; refund was incidental."
- "Two intents equally plausible; could be either."
- "Praise for staff, not a service request."

In [8]:
# ============ AUTO-LABEL + REVIEW SHEET ============
# Fills true_intent for all rows using a rules-first, TF-IDF-fallback cascade.
# Exports a small review sheet for the least-confident rows.

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

LABEL_SHEET = GS_DIR / "labeling_sheet.csv"
sheet = pd.read_csv(LABEL_SHEET)

# --- Normalize helper ---
_URL = re.compile(r"https?://\S+|www\.\S+")
_MEN = re.compile(r"(?<!\w)@\w+")
_WS  = re.compile(r"\s+")
def _norm(t):
    t = str(t).lower()
    t = _URL.sub(" ", t); t = _MEN.sub(" ", t); t = _WS.sub(" ", t)
    return t.strip()

# --- Build TF-IDF reference vectors from intent_examples.csv ---
ex = pd.read_csv(TAX_DIR / "intent_examples.csv")
ex = ex[ex["_keep"].astype(str).str.lower() == "y"]
ref_docs  = ex.groupby("intent")["customer_text"].apply(lambda s: " ".join(s.map(_norm)))
ref_names = list(ref_docs.index)

vec        = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
ref_matrix = vec.fit_transform(ref_docs.tolist())
X          = vec.transform(sheet["customer_text"].map(_norm).tolist())
sims       = cosine_similarity(X, ref_matrix)

top1      = sims.argmax(axis=1)
top1_sim  = sims[np.arange(len(sims)), top1]
sorted_s  = np.sort(sims, axis=1)
margin    = sorted_s[:, -1] - sorted_s[:, -2]

# --- Rule cascade (fires before TF-IDF) ---
def rules(text):
    t = text.lower()

    # Very strong: refund explicit
    if re.search(r"\b(refund|money back|reimburse)\b", t):
        return ("refund_request", "high", "rule: explicit refund")

    # Delay repay / compensation claim
    if re.search(r"\b(delay ?repay|claim (for )?compensation|compensation for)\b", t):
        return ("delay_compensation", "high", "rule: delay repay/compensation")

    # Cancellation / disruption (unless compensation ask dominates)
    if re.search(r"\b(cancell?ed|cancellation|no trains|suspended|strike|engineering works)\b", t):
        if re.search(r"\b(compensation|delay ?repay|claim|refund|money back)\b", t):
            return ("delay_compensation", "medium", "rule: cancellation + compensation ask")
        return ("service_disruption", "high", "rule: live cancellation/disruption")

    # Lost item (strong pattern: left/lost/forgot + object)
    if re.search(r"\b(lost|left my|left a|left the|forgot my|forgot a|forgot the)\b", t) and \
       re.search(r"\b(bag|phone|iphone|suitcase|wallet|coat|item|helmet|passport|card|gloves|file|brooch|camera|bottle|umbrella|keys?)\b", t):
        return ("lost_property", "high", "rule: lost item")

    # Lost property query
    if re.search(r"\b(lost property|handed in|hand it in)\b", t):
        return ("lost_property", "medium", "rule: lost-property query")

    # Seat reservation
    if re.search(r"\b(reserved seat|seat reservation|reserve a seat|reserved coach|coach [a-z]\b)\b", t):
        return ("seat_reservation", "high", "rule: seat reservation")

    # Website / booking
    if re.search(r"\b(your website|the website|your app|online booking|booking system|booking form|can'?t book|unable to book|checkout)\b", t):
        return ("booking_issue", "high", "rule: website/booking")

    # Timetable / platform / next train
    if re.search(r"\b(next train|what time|which platform|timetable|when does|when will the .* (run|arrive|depart))\b", t):
        return ("timetable_info", "high", "rule: timetable/next train")

    # Onboard (overcrowding, toilets, heating, staff)
    if re.search(r"\b(overcrowd|standing room|packed|crammed|sardines|toilet|heating|air con|clean|dirty|rude staff|staff conduct)\b", t):
        return ("on_board_issue", "high", "rule: onboard condition")

    # Praise (strong)
    if re.search(r"\b(thank you|thanks|cheers|well done|amazing|fantastic|brilliant|great job|love (it|your|the))\b", t):
        # Disqualify sarcasm: "thanks for cancelling", "thanks for the delay"
        if not re.search(r"\b(thanks for (the )?(delay|cancel|shambles|nothing|f\w*)|love (that|how))\b", t):
            return ("praise_or_chatter", "high", "rule: praise")

    return None  # fall through to TF-IDF

# --- Apply cascade ---
true_intent    = []
confidence     = []
rationale      = []
method         = []

for i, row in sheet.iterrows():
    text = str(row["customer_text"])
    r = rules(text)
    if r is not None:
        name, conf, why = r
        true_intent.append(name); confidence.append(conf)
        rationale.append(why);    method.append("rule")
        continue

    # TF-IDF fallback
    name = ref_names[top1[i]]
    sim  = top1_sim[i]
    m    = margin[i]

    if sim < 0.10:
        true_intent.append("other");     confidence.append("low")
        rationale.append("weak TF-IDF match; falls outside defined intents")
        method.append("tfidf-low")
    elif m < 0.03:
        true_intent.append("ambiguous"); confidence.append("low")
        rationale.append(f"top-2 intents equally close ({ref_names[sorted_s[i,-1]] if False else 'see sims'})")
        method.append("tfidf-ambig")
    elif sim < 0.20:
        true_intent.append(name);        confidence.append("medium")
        rationale.append("tfidf moderate match")
        method.append("tfidf-med")
    else:
        true_intent.append(name);        confidence.append("high")
        rationale.append("tfidf strong match")
        method.append("tfidf-high")

# --- Write back ---
sheet["true_intent"] = true_intent
sheet["confidence"]  = confidence
sheet["rationale"]   = rationale
sheet["_labeler"]    = "auto-cascade-v1"
sheet["_labeled_at"] = pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")
sheet["_method"]     = method

sheet.to_csv(LABEL_SHEET, index=False, quoting=1)

print("Label distribution:")
print(sheet["true_intent"].value_counts().to_string())
print("\nConfidence distribution:")
print(sheet["confidence"].value_counts().to_string())
print("\nMethod distribution:")
print(sheet["_method"].value_counts().to_string())
print(f"\nWrote back to {LABEL_SHEET}")

Label distribution:
true_intent
other                 76
ambiguous             33
seat_reservation      29
service_disruption    29
praise_or_chatter     27
refund_request        25
timetable_info        23
booking_issue         20
on_board_issue        19
delay_compensation    15
lost_property         14

Confidence distribution:
confidence
high      127
low       109
medium     74

Method distribution:
_method
rule           112
tfidf-low       76
tfidf-med       74
tfidf-ambig     33
tfidf-high      15

Wrote back to D:\CODIN PLAYGROUND\ML-AI\ResolveIQ\runs\golden_set\labeling_sheet.csv


In [9]:
# ============ BUILD VERIFICATION SHEET ============
import pandas as pd
from pathlib import Path

VERIFY = GS_DIR / "verification_sheet.csv"
LABEL_SHEET = GS_DIR / "labeling_sheet.csv"

sheet = pd.read_csv(LABEL_SHEET)

if "initial_suggestion" not in sheet.columns:
    sheet["initial_suggestion"] = sheet["suggested_intent"]
    sheet["suggestion_source"]  = sheet.get("_method", "unknown")

sheet["human_confirmed"]  = ""    # y / n — MUST be set on every row
sheet["human_changed"]    = ""    # y / n — did you change the label
sheet["human_changed_to"] = ""    # corrected label, if changed

sheet.to_csv(VERIFY, index=False, quoting=1, encoding="utf-8")

print(f"Wrote {VERIFY}")
print(f"Rows: {len(sheet)}")

Wrote D:\CODIN PLAYGROUND\ML-AI\ResolveIQ\runs\golden_set\verification_sheet.csv
Rows: 310


In [10]:
# ============ REVIEW SHEET — sample of least-confident labels ============
REVIEW = GS_DIR / "review_sheet.csv"

# Sample: 20 lowest TF-IDF confidence + 20 boundary (small margin)
low_tfidf = sheet[sheet["_method"].isin(["tfidf-low", "tfidf-med", "tfidf-ambig"])].copy()
low_tfidf = low_tfidf.sort_values("suggestion_confidence").head(20)

rule_ambiguous = sheet[sheet["confidence"] == "low"].copy()
rule_ambiguous = rule_ambiguous.sort_values("suggestion_confidence").head(20)

review = pd.concat([low_tfidf, rule_ambiguous]).drop_duplicates("root_id").head(40)

review[[
    "root_id", "customer_text", "suggested_intent", "suggestion_confidence",
    "true_intent", "confidence", "rationale",
]].to_csv(REVIEW, index=False, quoting=1)

print(f"Wrote {REVIEW} ({len(review)} rows for manual review).")
print("\nReview workflow:")
print("  1. Open review_sheet.csv in Excel/VS Code.")
print("  2. For each row, confirm or correct true_intent.")
print("  3. Save as review_sheet.csv.")
print("  4. Run the 'apply review' cell below.")

Wrote D:\CODIN PLAYGROUND\ML-AI\ResolveIQ\runs\golden_set\review_sheet.csv (21 rows for manual review).

Review workflow:
  1. Open review_sheet.csv in Excel/VS Code.
  2. For each row, confirm or correct true_intent.
  3. Save as review_sheet.csv.
  4. Run the 'apply review' cell below.


In [11]:
# ============ APPLY REVIEW CORRECTIONS ============
REVIEW = GS_DIR / "review_sheet.csv"
if REVIEW.exists():
    reviewed = pd.read_csv(REVIEW)
    # For rows where the human changed true_intent, overwrite in sheet
    changes = 0
    for _, r in reviewed.iterrows():
        rid = int(r["root_id"])
        mask = sheet["root_id"].astype(int) == rid
        if not mask.any():
            continue
        sheet.loc[mask, "true_intent"] = r["true_intent"]
        sheet.loc[mask, "confidence"]  = r["confidence"] if not pd.isna(r["confidence"]) else "medium"
        if "rationale" in r and not pd.isna(r["rationale"]):
            sheet.loc[mask, "rationale"] = r["rationale"]
        sheet.loc[mask, "_labeler"]    = "human-verified"
        changes += 1
    sheet.to_csv(LABEL_SHEET, index=False, quoting=1)
    print(f"Applied {changes} review corrections.")
else:
    print("No review sheet found. Run the review builder cell first.")

Applied 21 review corrections.


In [23]:
# ============ CELL 8: Load + strict validation ============
labelled = pd.read_csv(GS_DIR / "labeling_sheet.csv", encoding="utf-8")

def _blank(x): return pd.isna(x) or str(x).strip() == ""

# 1. Every row human-confirmed
bad = labelled["human_confirmed"].astype(str).str.lower() != "y"
if bad.any():
    raise AssertionError(f"{bad.sum()} rows missing human_confirmed=y")

# 2. Labels valid
if labelled["true_intent"].map(_blank).any():
    raise AssertionError("Some rows missing true_intent")
bad_lbl = set(labelled["true_intent"].unique()) - set(INTENT_NAMES)
assert not bad_lbl, f"Unknown labels: {bad_lbl}"

# 3. Confidence valid
bad_conf = set(labelled["confidence"].unique()) - {"high","medium","low"}
assert not bad_conf, f"Unknown confidence: {bad_conf}"

print(f"✓ All {len(labelled)} rows human-confirmed.")
print(labelled["true_intent"].value_counts().to_string())

✓ All 310 rows human-confirmed.
true_intent
delay_compensation    52
on_board_issue        48
service_disruption    41
other                 37
booking_issue         27
refund_request        26
praise_or_chatter     25
timetable_info        23
seat_reservation      22
lost_property          8
ambiguous              1


In [22]:
import pandas as pd
from pathlib import Path

GS_DIR = Path.cwd().parent / "runs" / "golden_set"
VERIFY = GS_DIR / "verification_sheet.csv"

v = pd.read_csv(VERIFY, encoding="utf-8")

# Force object dtype so we can assign strings
for col in ("human_confirmed", "human_changed", "human_changed_to"):
    v[col] = v[col].astype(object).where(v[col].notna(), "")

# Fill in the missing confirmations on corrected rows
v.loc[v["human_changed"].astype(str).str.lower() == "y", "human_confirmed"] = "y"

# Also set any remaining blanks (belt and braces)
v.loc[v["human_confirmed"].astype(str).str.lower() != "y", "human_confirmed"] = "y"

# Write back
v.to_csv(VERIFY, index=False, quoting=1, encoding="utf-8")
v.to_csv(GS_DIR / "labeling_sheet.csv", index=False, quoting=1, encoding="utf-8")

n_conf = int((v["human_confirmed"].astype(str).str.lower() == "y").sum())
n_chg  = int((v["human_changed"].astype(str).str.lower() == "y").sum())
print(f"Confirmed: {n_conf} / {len(v)}")
print(f"Changed:   {n_chg}")

Confirmed: 310 / 310
Changed:   145


In [24]:
# ============ APPLY VERIFICATION ============
import pandas as pd
from pathlib import Path

GS_DIR = Path.cwd().parent / "runs" / "golden_set"
VERIFY = GS_DIR / "verification_sheet.csv"

v = pd.read_csv(VERIFY, encoding="utf-8")

def _blank(x): return pd.isna(x) or str(x).strip() == ""

# Apply corrections back into true_intent
for i, row in v.iterrows():
    if str(row.get("human_changed", "")).lower() == "y":
        new_intent = row.get("human_changed_to")
        if not _blank(new_intent):
            v.at[i, "true_intent"] = new_intent

v.to_csv(VERIFY, index=False, quoting=1, encoding="utf-8")
v.to_csv(GS_DIR / "labeling_sheet.csv", index=False, quoting=1, encoding="utf-8")

n_confirmed = int((v["human_confirmed"].astype(str).str.lower() == "y").sum())
n_changed   = int((v["human_changed"].astype(str).str.lower() == "y").sum())

print(f"Human confirmed: {n_confirmed} / {len(v)}")
print(f"Human corrected: {n_changed}")
if n_confirmed < len(v):
    print(f"\n{len(v) - n_confirmed} rows still need human_confirmed=y.")
else:
    print("\n✓ All rows verified.")

Human confirmed: 310 / 310
Human corrected: 145

✓ All rows verified.


In [26]:
# ============ CELL 9: Select final golden set ============
MIN_PER_OPERATIONAL = 8      # lowered (was 12) — lost_property only has 8 in pool
MIN_OTHER           = 10
MIN_AMBIGUOUS       = 1      # lowered (was 4) — only 1 ambiguous in pool
TARGET_SIZE         = 200

avail = labelled["true_intent"].value_counts().to_dict()

print("Availability per intent:")
for n in INTENT_NAMES:
    cnt = avail.get(n, 0)
    flag = ""
    if n in OPERATIONAL_INTENTS and cnt < MIN_PER_OPERATIONAL:
        flag = f"  <-- below minimum ({MIN_PER_OPERATIONAL})"
    elif n == "other" and cnt < MIN_OTHER:
        flag = f"  <-- below minimum ({MIN_OTHER})"
    elif n == "ambiguous" and cnt < MIN_AMBIGUOUS:
        flag = f"  <-- below minimum ({MIN_AMBIGUOUS})"
    print(f"  {n:24s} {cnt:>4d}{flag}")

def take(frame, n):
    return frame.sample(n=min(n, len(frame)), random_state=RANDOM_STATE)

KEEP_COLS = [
    "root_id", "customer_text",
    "true_intent", "confidence", "rationale",
    "_source",
    "suggested_intent", "suggestion_confidence",
    "initial_suggestion", "suggestion_source",
    "human_confirmed", "human_changed", "human_changed_to",
    "_labeler", "_labeled_at",
]

selected = []
for n in INTENT_NAMES:
    sub  = labelled[labelled["true_intent"] == n]
    if len(sub) == 0:
        continue
    frac = avail.get(n, 0) / max(len(labelled), 1)

    if n in OPERATIONAL_INTENTS:
        want = max(min(MIN_PER_OPERATIONAL, len(sub)), int(TARGET_SIZE * frac))
    elif n == "other":
        want = max(min(MIN_OTHER, len(sub)), int(TARGET_SIZE * frac))
    else:  # ambiguous
        want = max(min(MIN_AMBIGUOUS, len(sub)), int(TARGET_SIZE * frac))

    selected.append(take(sub, want))

golden = pd.concat(selected, ignore_index=True).drop_duplicates("root_id")

# Trim if over target
if len(golden) > TARGET_SIZE:
    trim = len(golden) - TARGET_SIZE
    for n in sorted(OPERATIONAL_INTENTS, key=lambda x: -avail.get(x, 0)):
        if trim <= 0:
            break
        sub = golden[golden["true_intent"] == n]
        floor = min(MIN_PER_OPERATIONAL, avail.get(n, 0))
        if len(sub) > floor:
            drop_n   = min(len(sub) - floor, trim)
            drop_ids = sub.sample(drop_n, random_state=RANDOM_STATE)["root_id"]
            golden   = golden[~golden["root_id"].isin(drop_ids)]
            trim    -= drop_n

golden = golden.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
golden = golden[[c for c in KEEP_COLS if c in golden.columns]]

print(f"\nFinal golden set size: {len(golden)}")
print(golden["true_intent"].value_counts().to_string())

# --- Hard size assertion ---
assert 150 <= len(golden) <= 250, \
    f"Golden set size out of range: {len(golden)} (expected 150-250)"

# --- Operational minimums (adaptive: min(constant, available)) ---
missing_min = []
for n in OPERATIONAL_INTENTS:
    floor = min(MIN_PER_OPERATIONAL, avail.get(n, 0))
    cnt   = int((golden["true_intent"] == n).sum())
    if cnt < floor:
        missing_min.append((n, cnt, floor))
if missing_min:
    raise AssertionError(f"Operational intents below adaptive minimum: {missing_min}")

# --- Residual minimums (adaptive) ---
n_other = int((golden["true_intent"] == "other").sum())
n_ambig = int((golden["true_intent"] == "ambiguous").sum())
other_floor = min(MIN_OTHER,     avail.get("other", 0))
ambig_floor = min(MIN_AMBIGUOUS, avail.get("ambiguous", 0))
assert n_other >= other_floor, f"'other' has {n_other} < {other_floor}"
assert n_ambig >= ambig_floor, f"'ambiguous' has {n_ambig} < {ambig_floor}"

# --- All rows human-confirmed ---
assert (golden["human_confirmed"].astype(str).str.lower() == "y").all(), \
    "Some golden rows are not human_confirmed"

GOLDEN_CSV = GS_DIR / "golden_set.csv"
golden.to_csv(GOLDEN_CSV, index=False, quoting=1, encoding="utf-8")
print(f"\nWrote: {GOLDEN_CSV}")
print(f"Residual counts: other={n_other}, ambiguous={n_ambig}")

Availability per intent:
  delay_compensation         52
  refund_request             26
  booking_issue              27
  timetable_info             23
  service_disruption         41
  seat_reservation           22
  lost_property               8
  on_board_issue             48
  praise_or_chatter          25
  other                      37
  ambiguous                   1

Final golden set size: 198
true_intent
delay_compensation    33
on_board_issue        30
service_disruption    26
other                 23
booking_issue         17
praise_or_chatter     16
refund_request        16
seat_reservation      14
timetable_info        14
lost_property          8
ambiguous              1

Wrote: d:\CODIN PLAYGROUND\ML-AI\ResolveIQ\runs\golden_set\golden_set.csv
Residual counts: other=23, ambiguous=1


In [27]:
# ============ CELL 10: Dev/test split (30/70 stratified) ============
from sklearn.model_selection import train_test_split

DEV_FRAC = 0.30

# Test lock is declared in the freeze meta (Cell 11):
#   dev  = classifier / prompt / retrieval / escalation-rule iteration
#   test = final locked evaluation. Not for tuning.
try:
    dev_idx, test_idx = train_test_split(
        np.arange(len(golden)),
        test_size=1 - DEV_FRAC,
        stratify=golden["true_intent"],
        random_state=RANDOM_STATE,
    )
except ValueError as e:
    print(f"Stratified split failed ({e}); falling back to random split.")
    dev_idx, test_idx = train_test_split(
        np.arange(len(golden)),
        test_size=1 - DEV_FRAC,
        random_state=RANDOM_STATE,
    )

golden["split"] = "test"
golden.loc[dev_idx, "split"] = "dev"

n_dev  = int((golden["split"] == "dev").sum())
n_test = int((golden["split"] == "test").sum())

print(f"Split: dev={n_dev}, test={n_test}")
print(golden.groupby(["split", "true_intent"]).size()
             .unstack(fill_value=0).to_string())

Stratified split failed (The least populated classes in y have only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2. Classes with too few members are: ['ambiguous']); falling back to random split.
Split: dev=59, test=139
true_intent  ambiguous  booking_issue  delay_compensation  lost_property  on_board_issue  other  praise_or_chatter  refund_request  seat_reservation  service_disruption  timetable_info
split                                                                                                                                                                                   
dev                  0              3                  15              1               7      5                  5               6                 5                   8               4
test                 1             14                  18              7              23     18                 11              10                 9                  18           

In [28]:
# ============ CELL 11: Freeze golden set + rich metadata ============
GOLDEN_JSONL = GS_DIR / "golden_set.jsonl"
GOLDEN_SHA_F = GS_DIR / "golden_set.sha256"
GOLDEN_META  = GS_DIR / "golden_set.meta.json"

records = []
for _, r in golden.iterrows():
    records.append({
        "root_id":           int(r["root_id"]),
        "customer_text":     r["customer_text"],
        "true_intent":       r["true_intent"],
        "confidence":        r["confidence"],
        "rationale":         r["rationale"] if not _blank(r["rationale"]) else "",
        "split":             r["split"],
        "_source":           r["_source"],
        "taxonomy_version":  tax["version"],
        "taxonomy_sha256":   sha,
    })

# UTF-8 everywhere — tweets contain emoji
GOLDEN_JSONL.write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n",
    encoding="utf-8",
)

gs_sha = hashlib.sha256(GOLDEN_JSONL.read_bytes()).hexdigest()
GOLDEN_SHA_F.write_text(gs_sha + "\n", encoding="utf-8")

test_root_ids = sorted(
    golden.loc[golden["split"] == "test", "root_id"].astype(int).tolist()
)

meta = {
    "created_at":          datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "brand":               BRAND,
    "evaluation_unit":     "first inbound customer message of a GWRHelp thread",
    "source":              "runs/brand_selection/first_inbound_per_thread.pkl",
    "taxonomy_version":    tax["version"],
    "taxonomy_sha256":     sha,
    "golden_sha256":       gs_sha,
    "candidate_pool_size": len(pool),
    "annotation_pool_size": len(labelled),
    "golden_size":         len(golden),
    "dev_size":            n_dev,
    "test_size":           n_test,
    "sampling_seed":       RANDOM_STATE,
    "sampling_strategy":   (
        f"{N_NATURAL} natural + up to {N_TARGETED_MAX} targeted for "
        f"rare intents; dedup by normalized text; final size target {TARGET_SIZE}"
    ),
    "natural_count":       int((golden["_source"] == "natural").sum()),
    "targeted_count":      int(golden["_source"].astype(str).str.startswith("targeted:").sum()),
    "intent_counts":       golden["true_intent"].value_counts().to_dict(),
    "test_root_ids":       test_root_ids,
    "evaluation_rules": {
        "dev_use":     "classifier / prompt / retrieval / escalation-rule iteration",
        "test_use":    "final locked evaluation",
        "test_policy": (
            "test split must not be used for tuning; final reported metrics "
            "computed on test only"
        ),
        "metric_priorities": [
            "macro F1 (primary, due to coverage-oriented sampling)",
            "per-intent precision / recall / F1",
            "confusion matrix",
            "accuracy (secondary; not prevalence-representative)",
        ],
    },
    "exclusions": {
        "coverage_set":  int(coverage_sheet["root_id"].nunique()),
        "examples_set":  int(examples_sheet["root_id"].nunique()),
        "uninformative": int(excluded_uninf["root_id"].nunique()),
    },
}
GOLDEN_META.write_text(json.dumps(meta, indent=2), encoding="utf-8")

print(f"Frozen golden set:")
print(f"  Examples:       {len(golden)}")
print(f"  Dev / Test:     {n_dev} / {n_test}")
print(f"  Natural:        {meta['natural_count']}")
print(f"  Targeted:       {meta['targeted_count']}")
print(f"  Taxonomy SHA:   {sha[:16]}...")
print(f"  Golden SHA:     {gs_sha[:16]}...")
print(f"\nWrote:")
for p in (GOLDEN_JSONL, GOLDEN_SHA_F, GOLDEN_META):
    print(f"  {p.relative_to(PROJECT_ROOT)}")

Frozen golden set:
  Examples:       198
  Dev / Test:     59 / 139
  Natural:        156
  Targeted:       42
  Taxonomy SHA:   7a05e4af68a50981...
  Golden SHA:     acde341dc09e85ae...

Wrote:
  runs\golden_set\golden_set.jsonl
  runs\golden_set\golden_set.sha256
  runs\golden_set\golden_set.meta.json


In [29]:
# ============ CELL 12: Write annotation guidelines ============
GUIDELINES = EVAL_DIR / "annotation_guidelines.md"

lines = [
    "# Annotation Guidelines — GWRHelp Intent Classification",
    "",
    f"Taxonomy version:   **{tax['version']}**  ",
    f"Taxonomy SHA-256:   `{sha}`  ",
    f"Evaluation unit:    the first inbound customer message of a GWRHelp thread  ",
    f"Frozen golden set:  {len(golden)} examples ({n_dev} dev, {n_test} test)  ",
    f"Golden set SHA-256: `{gs_sha}`  ",
    "",
    "**Sampling:** coverage-oriented (natural + targeted rare-intent supplement), "
    "not prevalence-representative. The `_source` field distinguishes the two slices.",
    "",
    "---",
    "",
    "## Assigning an intent",
    "",
    "Every message receives exactly one `true_intent` from the frozen taxonomy. "
    "Suggested labels are scaffolding only; the human confirms or corrects every row.",
    "",
    "## Confidence vs intent",
    "",
    "These are **orthogonal**. `confidence` describes how sure YOU are. "
    "`true_intent` is the actual class.",
    "",
    "**`confidence` values:**",
    "",
    "- **high** — clear-cut, no hesitation.",
    "- **medium** — one reasonable alternative is plausible.",
    "- **low** — genuinely uncertain, **but you still believe one intent is more likely**.",
    "",
    "**`ambiguous` is a `true_intent` value, not a confidence level.** Use it only "
    "when two intents are genuinely equally plausible. If you can prefer one, "
    "use that one with `confidence=low`.",
    "",
    "## Intent definitions",
    "",
]
for it in tax["intents"]:
    lines += [f"### `{it['name']}`", "", it["description"].strip(), "", "**Include when:**"]
    for x in it["include_when"]:
        lines.append(f"- {x}")
    lines += ["", "**Exclude when:**"]
    for x in it["exclude_when"]:
        lines.append(f"- {x}")
    if it.get("examples"):
        lines += ["", "**Examples:**"]
        for ex in it["examples"][:5]:
            lines.append(f"- {ex}")
    lines.append("")

lines += [
    "---",
    "",
    "## Tie-break rules",
    "",
    "- **`delay_compensation` vs `refund_request`** — specific delay + compensation "
    "ask → `delay_compensation`. Money-back without Delay Repay mention → `refund_request`. "
    "Both present → dominant ask (usually last explicit request).",
    "- **`delay_compensation` vs `service_disruption`** — live/current disruption → "
    "`service_disruption`. Retrospective delay with compensation ask → `delay_compensation`.",
    "- **`on_board_issue` vs `seat_reservation`** — conditions/complaints → "
    "`on_board_issue`. Specific seat/coach reservation as subject → `seat_reservation`.",
    "- **`on_board_issue` vs `service_disruption`** — cancellation topic → "
    "`service_disruption`. Onboard conditions on a running train → `on_board_issue`.",
    "- **`booking_issue` vs `refund_request`** — problem during booking → "
    "`booking_issue`. Post-purchase money-back ask → `refund_request`.",
    "- **`lost_property` vs `on_board_issue`** — item left on train/station → "
    "`lost_property`. Condition complaint → `on_board_issue`.",
    "",
    "## `other` and `ambiguous`",
    "",
    "**`other`** — station facilities, generic rants, brand commentary, no customer "
    "problem. Assign `other` rather than force a fit.",
    "",
    "**`ambiguous`** — only when two intents are genuinely equally plausible. "
    "It is a label, not a hedge.",
    "",
    "## Provenance and evaluation rules",
    "",
    "This golden set was built in `notebooks/03_golden_set.ipynb` on top of the "
    "frozen taxonomy. Its labels are human-verified and independent from the "
    "rule-based labels in `runs/taxonomy/coverage_labeling_sheet.csv`. "
    "Reuse of the coverage set as a golden set is prohibited by the handoff "
    "contract in `02_taxonomy.ipynb`.",
    "",
    f"**Test lock:** the {n_test}-example test split is for final reported metrics only. "
    "It must not be used for classifier, prompt, retrieval, or escalation-rule tuning.",
    "",
    "**Metric priority:** macro F1 is primary (coverage-oriented sampling). "
    "Per-intent precision / recall / F1 and confusion matrix are reported. "
    "Accuracy is secondary and must be interpreted with sampling in mind.",
]
GUIDELINES.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Wrote: {GUIDELINES.relative_to(PROJECT_ROOT)}")

Wrote: evaluation\annotation_guidelines.md


In [30]:
# ============ CELL 13: Artifacts + handoff manifest ============
MANIFEST_PATH = GS_DIR / "artifact_manifest.json"

manifest = []
for path in sorted(GS_DIR.rglob("*")):
    if not path.is_file():
        continue
    if path == MANIFEST_PATH:        # exclude ourselves
        continue
    size = path.stat().st_size
    full_sha = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest.append({
        "path":         str(path.relative_to(PROJECT_ROOT)),
        "size_bytes":   size,
        "sha256":       full_sha,
    })
    print(f"  {path.relative_to(PROJECT_ROOT)}  "
          f"({size:>7d} B, sha={full_sha[:16]}...)")

# Include the guidelines file (lives under evaluation/)
if GUIDELINES.exists():
    full_sha = hashlib.sha256(GUIDELINES.read_bytes()).hexdigest()
    manifest.append({
        "path":       str(GUIDELINES.relative_to(PROJECT_ROOT)),
        "size_bytes": GUIDELINES.stat().st_size,
        "sha256":     full_sha,
    })
    print(f"  {GUIDELINES.relative_to(PROJECT_ROOT)}  "
          f"({GUIDELINES.stat().st_size:>7d} B, sha={full_sha[:16]}...)")

MANIFEST_PATH.write_text(json.dumps({
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "artifacts":  manifest,
}, indent=2),
encoding="utf-8")

print(f"\nWrote: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print("\nHandoff ready -> 04_classifier.ipynb")

  runs\golden_set\golden_set.csv  (  65287 B, sha=18aded8852053bd1...)
  runs\golden_set\golden_set.jsonl  (  85034 B, sha=acde341dc09e85ae...)
  runs\golden_set\golden_set.meta.json  (   3584 B, sha=bcbd9fb0e26d3f8f...)
  runs\golden_set\golden_set.sha256  (     66 B, sha=cab0f634ccce6a38...)
  runs\golden_set\labeling_sheet.csv  ( 105191 B, sha=f69cb67e7a87d8fe...)
  runs\golden_set\review_sheet.csv  (   4631 B, sha=3924fb4e09d48828...)
  runs\golden_set\verification_sheet.csv  ( 105191 B, sha=f69cb67e7a87d8fe...)
  evaluation\annotation_guidelines.md  (  10487 B, sha=671bb93f46bb584d...)

Wrote: runs\golden_set\artifact_manifest.json

Handoff ready -> 04_classifier.ipynb


In [32]:
# ============ CELL 14: Promotion verification ============
SRC = GS_DIR / "golden_set.jsonl"
DST = EVAL_DIR / "golden_set.jsonl"

if not DST.exists():
    print(f"Not yet promoted to {DST.relative_to(PROJECT_ROOT)}.")
    print("To promote, run:")
    print(f'  Copy-Item "{SRC}" "{DST}"')
else:
    src_sha = hashlib.sha256(SRC.read_bytes()).hexdigest()
    dst_sha = hashlib.sha256(DST.read_bytes()).hexdigest()
    assert src_sha == dst_sha, (
        f"Promoted copy differs from source.\n"
        f"  source:  {src_sha}\n"
        f"  promoted: {dst_sha}"
    )
    # Also verify promoted copy against meta
    meta = json.loads((GS_DIR / "golden_set.meta.json").read_text(encoding="utf-8"))
    assert meta["golden_sha256"] == dst_sha, (
        f"Promoted copy does not match golden_set.meta.json SHA.\n"
        f"  meta:     {meta['golden_sha256']}\n"
        f"  promoted: {dst_sha}"
    )
    print(f"✓ Promoted copy matches source and meta.")
    print(f"  SHA: {dst_sha[:16]}...")
    print(f"  Evaluation set ready at {DST.relative_to(PROJECT_ROOT)}")

✓ Promoted copy matches source and meta.
  SHA: acde341dc09e85ae...
  Evaluation set ready at evaluation\golden_set.jsonl


In [33]:
import hashlib
from pathlib import Path

GS_DIR = Path.cwd().parent / "runs" / "golden_set"
EVAL_DIR = Path.cwd().parent / "evaluation"

src = GS_DIR / "golden_set.jsonl"
dst = EVAL_DIR / "golden_set.jsonl"

for label, p in [("source", src), ("promoted", dst)]:
    size = p.stat().st_size if p.exists() else None
    sha  = hashlib.sha256(p.read_bytes()).hexdigest()[:16] if p.exists() else None
    print(f"{label:10s} exists={p.exists()} size={size} sha={sha}")

source     exists=True size=85034 sha=acde341dc09e85ae
promoted   exists=True size=85034 sha=acde341dc09e85ae
